# RamanBench — HPO and Ensemble Ablation

> **Not yet published.** This notebook will be released alongside the HPO/ensemble results.

This notebook compares three benchmark conditions to quantify the contribution of
hyperparameter optimisation and ensemble methods on top of the default single-fit baseline.

| Condition | `optimize` | `ensemble` | Description |
|---|---|---|---|
| **default** | ✗ | ✗ | Single fit, default hyperparameters |
| **hpo** | ✓ | ✗ | Bayesian HPO within 4 h budget, best model wins |
| **hpo + ensemble** | ✓ | ✓ | Same HPO + 8-fold bagging across all trials |

**Prerequisites:** complete the cluster runs described in notebook 05 for all three
conditions before running the cells below.

In [ ]:
/Users/koddenbrock/Repository/raman_bench_paper/notebooks

## 1 — Submit the HPO and ensemble jobs

If the default condition is already done, submit only the remaining two:

```bash
cd $BASE/raman_bench_paper

bash cluster/submit_v1_full.sh --hpo-only
bash cluster/submit_v1_full.sh --ensemble-only

# Monitor
squeue -u $USER
find results/v1_hpo          -name '*.csv' | wc -l
find results/v1_hpo_ensemble -name '*.csv' | wc -l
```

## 2 — Compute metrics for all three conditions

In [ ]:
import subprocess, sys, json

configs = {
    "default":      "configs/v1_default.json",
    "hpo":          "configs/v1_hpo.json",
    "hpo_ensemble": "configs/v1_hpo_ensemble.json",
}

for condition, config_path in configs.items():
    result = subprocess.run(
        [sys.executable, "scripts/run_benchmark.py",
         "--config", config_path,
         "--step",   "metrics"],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else "FAILED"
    print(f"{condition:>14}: {status}")
    if result.returncode != 0:
        print(result.stderr[-500:])

## 3 — Load leaderboards

In [ ]:
import pandas as pd
from pathlib import Path
from raman_bench import Leaderboard

leaderboards = {}
for condition, config_path in configs.items():
    cfg = json.load(open(config_path))
    metrics_dir = Path(cfg["output_dir"]) / "metrics"
    reg_path = metrics_dir / "regression_metrics.csv"
    clf_path = metrics_dir / "classification_metrics.csv"
    if not reg_path.exists():
        print(f"{condition}: metrics not found, skipping")
        continue
    lb = Leaderboard(
        reg_metrics=pd.read_csv(reg_path),
        clf_metrics=pd.read_csv(clf_path) if clf_path.exists() else pd.DataFrame(),
    )
    leaderboards[condition] = lb
    print(f"{condition:>14}: loaded")

print(f"\n{len(leaderboards)}/3 conditions available")

## 4 — Per-condition rankings

In [ ]:
import matplotlib.pyplot as plt

n = len(leaderboards)
fig, axes = plt.subplots(1, n, figsize=(7 * n, 8), sharey=False)
if n == 1:
    axes = [axes]

for ax, (condition, lb) in zip(axes, leaderboards.items()):
    lb.plot(task="overall", n_top=29, ax=ax)
    ax.set_title(condition)

plt.tight_layout()
plt.show()

## 5 — Condition comparison: HPO gain and ensemble gain

In [ ]:
if len(leaderboards) == 3:
    boards = {}
    for cond, lb in leaderboards.items():
        r = lb.rank()[["Model", "Score", "Avg Rank"]].copy()
        r.columns = ["Model", f"Score ({cond})", f"Avg Rank ({cond})"]
        boards[cond] = r.set_index("Model")

    comparison = boards["default"].join(boards["hpo"]).join(boards["hpo_ensemble"])
    comparison["HPO gain"]      = comparison["Score (hpo)"]          - comparison["Score (default)"]
    comparison["Ensemble gain"] = comparison["Score (hpo_ensemble)"] - comparison["Score (hpo)"]
    comparison.sort_values("Score (hpo_ensemble)", ascending=False)

In [ ]:
if len(leaderboards) == 3:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, col, title in zip(
        axes,
        ["HPO gain", "Ensemble gain"],
        ["Score gain: default → HPO", "Score gain: HPO → HPO + ensemble"],
    ):
        data = comparison[col].sort_values(ascending=True)
        colors = ["#2196F3" if v >= 0 else "#F44336" for v in data]
        data.plot.barh(ax=ax, color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("ΔScore")

    plt.tight_layout()
    plt.show()